# 1. Install & Setup

Show how to install the openai package and set environment variables for API keys and defaults using bash commands.

In [ ]:
pip install -U openai
# Your repo/package with the refactor installed/available
# export environment variables:
export OPENAI_API_KEY=sk-...
# optional: override defaults
export OPENAI_DEFAULT_MODEL=gpt-4o
export OPENAI_TEMPERATURE=0.2

# 2. Import Required Libraries

Import functions and classes from openai_services, including get_text, get_structured, chat, call_with_tools, RollingTokenHistory, and ToolSpec.

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

from openai_service.chat_util import get_text, get_structured, chat, call_with_tools
from openai_service.history import RollingTokenHistory
from openai_service.tools import ToolSpec

# 3. Plain Text Call Example

Demonstrate a minimal call to get_text with a single prompt and print the response text.

In [ ]:
res = get_text("Write a joke about cats.")
print(res["text"])

# 4. Multiple Prompts Example

Show how to pass a list of prompts to get_text for multiple user turns and print the result.

In [ ]:
res = get_text([
    "You are a helpful explainer.",
    "Now, summarize K-means in 2 lines.",
    "Write a joke about it"
])
print(res["text"])

# 5. Mixed Text and Image Input

Demonstrate passing both text and an image URL to get_text and printing the captioned output.

In [ ]:
img_url = "https://www.angelosbakery.com/wp-content/uploads/2023/09/4-6-charcuterie-board.jpg"
res = get_text(["Label everything in this charcuterie board", img_url])
print(res["text"])

# 6. Structured Conversation Object

Show usage of a conversation dictionary with system, developer, history, and user keys, then call get_text and print the result.

In [4]:
conversation = {
    "system": "You are terse and precise.",
    "developer": "If the user asks for code, prefer Python.",
    "history": [
        {"role": "user", "content": [{"type":"input_text", "text":"Give me 3 facts about Mars"}]},
        {"role": "assistant", "content": [{"type":"output_text", "text":"• It's dusty\n• It's cold\n• It's photogenic"}]},
    ],
    "user": "Now give me 1 more fact."
}
res = get_text(conversation)
print(res["text"])

Sending messages: [{'role': 'system', 'content': [{'type': 'input_text', 'text': 'You are terse and precise.'}]}, {'role': 'developer', 'content': [{'type': 'input_text', 'text': 'If the user asks for code, prefer Python.'}]}, {'role': 'user', 'content': [{'type': 'input_text', 'text': 'Give me 3 facts about Mars'}]}, {'role': 'assistant', 'content': [{'type': 'output_text', 'text': "• It's dusty\n• It's cold\n• It's photogenic"}]}, {'role': 'user', 'content': [{'type': 'input_text', 'text': 'Now give me 1 more fact.'}]}]
• It has the largest volcano in the solar system, Olympus Mons.


# 7. Structured Outputs with JSON Schema

Define a JSON schema and use get_structured to request a structured output, handling validation and printing the parsed JSON.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "topics": {"type": "array", "items": {"type": "string"}},
        "estimated_minutes": {"type": "integer", "minimum": 1}
    },
    "required": ["title", "topics"],
    "additionalProperties": False
}

res = get_structured("Create a study plan about decision trees.", schema=schema)
plan = res["json"]     # already parsed & validated
print(plan["title"], plan["topics"])

# 8. Custom Messages with Roles

Demonstrate building a list of message dicts with explicit roles and using chat to get a response.

In [ ]:
msgs = [
    {"role":"system","content":[{"type":"input_text","text":"You are brief."}]},
    {"role":"user","content":[{"type":"input_text","text":"List 3 fast sorting algorithms"}]},
]
res = chat(msgs)
print(res["text"])

# 9. Rolling History Window

Show how to use RollingTokenHistory to maintain conversation history across multiple get_text calls.

In [ ]:
history = RollingTokenHistory(max_turns=8)  # simple heuristic by turns

res = get_text(
    ["We’ll talk a lot.", "Remember I prefer examples."],
    history=history
)

res = get_text(
    "Now explain Bloom filters with an example.",
    history=history
)

print(res["text"])

# 10. Manual Tool Calling

Define a ToolSpec and a handler function, use call_with_tools to get tool calls, manually execute the tool, and pass results back to chat.

In [ ]:
import json
from datetime import datetime
from openai_services import call_with_tools
from openai_services.tools import ToolSpec

def weather_lookup(args):
    # pretend to call a real weather API
    city = args["city"]
    units = args.get("units", "F")
    return {"city": city, "units": units, "temp": 97, "as_of": datetime.utcnow().isoformat() + "Z"}

weather_tool = ToolSpec(
    name="get_weather",
    description="Get current weather by city name.",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type":"string"},
            "units": {"type":"string", "enum":["F","C"]}
        },
        "required": ["city"],
        "additionalProperties": False
    }
    # handler omitted: we'll execute manually in this example
)

prompt = "What's the weather in Dallas right now? Return the tool arguments only."
res = call_with_tools(prompt, tools=[weather_tool])

print("Tool calls:", res["tool_calls"])     # → [{"id","name","args":{...}}]
print("Model follow-up (if any):", res["follow_up"])

# MANUAL EXECUTION:
calls = res["tool_calls"]
if calls:
    result = weather_lookup(calls[0]["args"])
    # Pass the result back as a tool message and ask for a final summary:
    tool_msg = {
        "role": "tool",
        "tool_call_id": calls[0]["id"],
        "content": [{"type":"output_text","text": json.dumps(result)}]
    }
    # Build a new conversation and call get_text/chat again:
    final = chat([
        {"role":"user","content":[{"type":"input_text","text": prompt}]},
        tool_msg
    ])
    print("Final:", final["text"])